# 03b — SVM regression-then-threshold baseline (Kachuee 2015)

Two `SVR` models — one for SBP, one for DBP — trained on the 10 `KACHUEE_FEATURES`, then thresholded into the AHA 3-class label. This is the canonical regression-then-threshold baseline the writeup compares against the direct multi-class classifiers (RF / XGBoost / 1D CNN). Subject-level splits are reused from `splits.json` so the test set matches the other models exactly.

**NaN handling**: AI / LASI / S1–S4 require detecting the dicrotic notch and diastolic peak per cycle and are NaN on ~84% of segments at 125 Hz. Following the strategy in `SVM_BASELINE.md`, we **median-impute** inside the sklearn pipeline (fit on train, applied to test) so the SVM sees the full ~80% of segments that have at least PTT + HR — rather than dropping to ~16%.

**Soft probabilities for AUROC**: regression outputs don't give native class probabilities. We construct them from the AHA-threshold distances: `sigmoid((sbp_pred - 130)/10)` etc. Documented in the commit message.

Outputs (the four files `notebooks/04_results.ipynb` reads):

- `models/svm_regression.joblib` — dict `{'sbp': pipeline, 'dbp': pipeline}`
- `data/processed/svm_metrics.json` — `MultiClassMetrics`
- `data/processed/svm_regression_metrics.json` — SBP/DBP/MAP MAE+STD plus best CV params
- `data/processed/svm_predictions.csv` — per-segment predictions

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import dataclasses
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

from bme_ml.paths import setup_paths
from bme_ml.splits import Splits, select_rows, add_subject_id
from bme_ml.labels import add_multiclass_column, multiclass_label, MULTICLASS_NAMES
from bme_ml.features import KACHUEE_FEATURES
from bme_ml.evaluation import evaluate_multiclass

paths = setup_paths()
print('KACHUEE_FEATURES:', KACHUEE_FEATURES)

KACHUEE_FEATURES: ['ptt_ms', 'pttf_ms', 'pttd_ms', 'hr_bpm', 'ai_kachuee', 'lasi_ms', 's1_area', 's2_area', 's3_area', 's4_area']


In [2]:
SVM_TRAIN_SUBSAMPLE = 10000   # SVR is O(n²) memory; cap train set. None = no cap.
SUBSAMPLE_SEED = 42

features = add_subject_id(pd.read_parquet(paths.features_parquet))
features = add_multiclass_column(features)
splits = Splits.from_json(paths.splits_json)

train_subjects = sorted({s for f in splits.cv_folds for s in f['train'] + f['val']})
# Drop only on the regression targets; the SVM pipeline imputes the
# landmark-dependent features so we keep ~80% of segments rather than ~16%.
train_df = select_rows(features, train_subjects).dropna(subset=['sbp', 'dbp']).reset_index(drop=True)
test_df  = select_rows(features, splits.test_subjects).dropna(subset=['sbp', 'dbp']).reset_index(drop=True)

# SVR with RBF kernel needs an n×n Gram matrix; on the full dataset (~120k
# train rows) that's ~115 GB. Subsample stratified by subject so every
# train subject still contributes at least one row when possible.
if SVM_TRAIN_SUBSAMPLE is not None and len(train_df) > SVM_TRAIN_SUBSAMPLE:
    rng = np.random.default_rng(SUBSAMPLE_SEED)
    n_subj = train_df['subject_id'].nunique()
    per_subject = max(1, SVM_TRAIN_SUBSAMPLE // n_subj)
    sampled = (
        train_df.groupby('subject_id', group_keys=False)
                .apply(lambda g: g.sample(min(len(g), per_subject), random_state=int(rng.integers(0, 2**31))))
    )
    if len(sampled) > SVM_TRAIN_SUBSAMPLE:
        sampled = sampled.sample(SVM_TRAIN_SUBSAMPLE, random_state=SUBSAMPLE_SEED)
    print(f'subsampled train: {len(train_df)} -> {len(sampled)} '
          f'(stratified by {n_subj} subjects, ~{per_subject}/subject)')
    train_df = sampled.reset_index(drop=True)

X_train = train_df[KACHUEE_FEATURES].to_numpy(np.float32)
X_test  = test_df[KACHUEE_FEATURES].to_numpy(np.float32)
groups_train = train_df['subject_id'].to_numpy()

print(f'train: {X_train.shape}  test: {X_test.shape}')
print(f'train subjects: {train_df["subject_id"].nunique()}  '
      f'test subjects: {test_df["subject_id"].nunique()}')
print('per-feature non-null in train:')
for c in KACHUEE_FEATURES:
    print(f'  {c:14s} {train_df[c].notna().mean():.1%}')
print('class distribution (test):')
print(test_df['label_3class'].value_counts().rename(index=dict(enumerate(MULTICLASS_NAMES))))

subsampled train: 102457 -> 9589 (stratified by 9589 subjects, ~1/subject)
train: (9589, 10)  test: (25687, 10)
train subjects: 9589  test subjects: 2398
per-feature non-null in train:
  ptt_ms         57.1%
  pttf_ms        51.7%
  pttd_ms        41.5%
  hr_bpm         100.0%
  ai_kachuee     64.8%
  lasi_ms        64.8%
  s1_area        64.8%
  s2_area        64.8%
  s3_area        64.8%
  s4_area        64.8%
class distribution (test):
label_3class
Hypertensive    12386
Normal           9335
Elevated         3966
Name: count, dtype: int64


C:\Users\Paarth\AppData\Local\Temp\ipykernel_48016\3653633048.py:23: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), per_subject), random_state=int(rng.integers(0, 2**31))))


In [3]:
param_grid = {
    'svr__C':     [0.1, 1, 10, 100],
    'svr__gamma': [1e-3, 1e-2, 1e-1, 1, 'scale'],
}

n_subjects = len(set(groups_train))
n_splits = min(5, n_subjects)
gkf = GroupKFold(n_splits=n_splits)
cv_splits = list(gkf.split(X_train, groups=groups_train))
print(f'GroupKFold: {n_splits} folds over {n_subjects} subjects')

svm_pipelines = {}
best_params   = {}
for target in ('sbp', 'dbp'):
    y_train = train_df[target].to_numpy(np.float32)
    pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('svr',     SVR(kernel='rbf')),
    ])
    grid = GridSearchCV(
        pipe, param_grid=param_grid,
        cv=cv_splits,
        scoring='neg_mean_absolute_error',
        n_jobs=-1, verbose=1, refit=True,
    )
    grid.fit(X_train, y_train)
    svm_pipelines[target] = grid.best_estimator_
    best_params[target]   = grid.best_params_
    print(f'{target}: best={grid.best_params_}, CV MAE={-grid.best_score_:.2f} mmHg')

GroupKFold: 5 folds over 9589 subjects
Fitting 5 folds for each of 20 candidates, totalling 100 fits


sbp: best={'svr__C': 10, 'svr__gamma': 1}, CV MAE=16.21 mmHg
Fitting 5 folds for each of 20 candidates, totalling 100 fits


dbp: best={'svr__C': 10, 'svr__gamma': 1}, CV MAE=8.05 mmHg


In [4]:
sbp_pred = svm_pipelines['sbp'].predict(X_test)
dbp_pred = svm_pipelines['dbp'].predict(X_test)

y_pred = np.fromiter(
    (multiclass_label(s, d) for s, d in zip(sbp_pred, dbp_pred)),
    dtype=np.int64, count=len(sbp_pred),
)
y_true = test_df['label_3class'].to_numpy(np.int64)

# Soft probabilities from regression outputs — logistic of distance from
# the AHA thresholds (scale = 10 mmHg for SBP, 5 mmHg for DBP). Lets the
# AUROC reflect the regressor's confidence rather than collapsing to 0.5.
def sigmoid(x): return 1.0 / (1.0 + np.exp(-x))

p_hyper_sbp  = sigmoid((sbp_pred - 130.0) / 10.0)
p_hyper_dbp  = sigmoid((dbp_pred -  80.0) /  5.0)
p_hyper      = 1.0 - (1.0 - p_hyper_sbp) * (1.0 - p_hyper_dbp)  # OR — matches AHA def.
p_above_120  = sigmoid((sbp_pred - 120.0) /  5.0)
p_elev       = p_above_120 * (1.0 - p_hyper)
p_normal     = (1.0 - p_above_120) * (1.0 - p_hyper)

y_proba = np.stack([p_normal, p_elev, p_hyper], axis=1).astype(np.float32)
y_proba = y_proba / y_proba.sum(axis=1, keepdims=True).clip(min=1e-12)

metrics = evaluate_multiclass(y_true, y_pred, y_proba)
print(metrics)

MultiClassMetrics(accuracy=0.4206407910616265, f1_macro=0.41369793248249703, confusion=[[3452, 4158, 1725], [751, 2116, 1099], [1266, 5883, 5237]], hypertensive_auroc=0.6822462190414533, hypertensive_pr_auc=0.6448466399554894, hypertensive_recall=0.42281608267398674, hypertensive_false_negative_rate=0.5771839173260133)


In [5]:
(paths.processed / 'svm_metrics.json').write_text(
    json.dumps(dataclasses.asdict(metrics), indent=2))

map_pred = (sbp_pred + 2 * dbp_pred) / 3.0
map_true = (test_df['sbp'].to_numpy() + 2 * test_df['dbp'].to_numpy()) / 3.0
reg_metrics = {
    'sbp_mae':     float(np.abs(sbp_pred - test_df['sbp']).mean()),
    'sbp_std':     float((sbp_pred - test_df['sbp']).std()),
    'dbp_mae':     float(np.abs(dbp_pred - test_df['dbp']).mean()),
    'dbp_std':     float((dbp_pred - test_df['dbp']).std()),
    'map_mae':     float(np.abs(map_pred - map_true).mean()),
    'map_std':     float((map_pred - map_true).std()),
    'best_params': best_params,
}
(paths.processed / 'svm_regression_metrics.json').write_text(
    json.dumps(reg_metrics, indent=2))

pd.DataFrame({
    'row_index':         test_df.index,
    'sbp_pred':          sbp_pred,
    'dbp_pred':          dbp_pred,
    'label_3class_pred': y_pred,
}).to_csv(paths.processed / 'svm_predictions.csv', index=False)

joblib.dump(svm_pipelines, paths.models / 'svm_regression.joblib')

print('Wrote:')
for name in ('svm_metrics.json', 'svm_regression_metrics.json', 'svm_predictions.csv'):
    p = paths.processed / name
    print(f'  {p}  ({p.stat().st_size} bytes)')
p = paths.models / 'svm_regression.joblib'
print(f'  {p}  ({p.stat().st_size} bytes)')
print('\nReg vs Kachuee Table I (full UCI, no subject-level split):')
print(f"  SBP MAE  {reg_metrics['sbp_mae']:.2f} mmHg  (Kachuee: 12.38)")
print(f"  DBP MAE  {reg_metrics['dbp_mae']:.2f} mmHg  (Kachuee:  6.34)")
print(f"  MAP MAE  {reg_metrics['map_mae']:.2f} mmHg  (Kachuee:  7.52)")

Wrote:
  C:\Users\Paarth\repos\bme-ml-bp-classification\data\processed\svm_metrics.json  (452 bytes)
  C:\Users\Paarth\repos\bme-ml-bp-classification\data\processed\svm_regression_metrics.json  (358 bytes)
  C:\Users\Paarth\repos\bme-ml-bp-classification\data\processed\svm_predictions.csv  (1165277 bytes)
  C:\Users\Paarth\repos\bme-ml-bp-classification\models\svm_regression.joblib  (1906640 bytes)

Reg vs Kachuee Table I (full UCI, no subject-level split):
  SBP MAE  16.43 mmHg  (Kachuee: 12.38)
  DBP MAE  8.10 mmHg  (Kachuee:  6.34)
  MAP MAE  9.54 mmHg  (Kachuee:  7.52)
